# 02 — Silver Cleaning & Data Quality

Bronze JSONL → audited Silver Parquet + DQ reports. **No Gold target. No modeling.**

- Event absence ≠ missing data (e.g. no pit rows → handled in Gold).
- Outliers flagged; only domain-invalid values removed.
- Run after Bronze with the **same** `USE_GOOGLE_DRIVE` setting as notebook 01.


## Colab setup (required every session)

Identical to `00` and `01`: clone, `pip install -e .`, Drive mount, then import `openf1_pipeline`.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Bronze prerequisites


In [2]:
from pathlib import Path

from openf1_pipeline.config import (
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.silver.build_silver import run_silver_cleaning

BRONZE_DIR = get_bronze_dir()
SILVER_DIR = get_silver_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
MANIFESTS_DIR = get_manifests_dir()

print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", BRONZE_DIR)
print("SILVER_DIR:", SILVER_DIR)
print("DATA_QUALITY_REPORTS_DIR:", DATA_QUALITY_REPORTS_DIR)

jsonl_files = list(BRONZE_DIR.rglob("*.jsonl")) if BRONZE_DIR.is_dir() else []
manifest_path = MANIFESTS_DIR / "ingestion_manifest.csv"
row_counts_path = DATA_QUALITY_REPORTS_DIR / "bronze_row_counts.csv"

print(f"Bronze JSONL files found: {len(jsonl_files)}")
print("ingestion_manifest.csv:", manifest_path.exists(), manifest_path)
print("bronze_row_counts.csv:", row_counts_path.exists(), row_counts_path)

if not jsonl_files:
    raise FileNotFoundError(
        f"Bronze data not found at {BRONZE_DIR}. "
        "Run 01_ingestion_bronze.ipynb first with the same USE_GOOGLE_DRIVE setting."
    )


OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
DATA_QUALITY_REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
Bronze JSONL files found: 16
ingestion_manifest.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/ingestion_manifest.csv
bronze_row_counts.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_row_counts.csv


## Run Silver cleaning


In [3]:
outputs = run_silver_cleaning(
    bronze_dir=BRONZE_DIR,
    silver_dir=SILVER_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
)
outputs["summary"]


INFO:openf1_pipeline.silver.build_silver:Loaded Bronze meetings: 25 rows from 1 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze sessions: 123 rows from 1 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze drivers: 40 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze laps: 2031 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze pit: 62 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze weather: 303 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze position: 927 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze race_control: 139 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze session_result: 40 rows from 2 files
INFO:openf1_pipeline.silver.build_silver:Loaded Bronze starting_grid: 0 rows from 0 files
INFO:openf1_pipeline.quality.duplicates:Skipped key duplicate check for starting_grid: missing columns ['session_key', 'driver_numb

{'tables_loaded': 10,
 'total_rows_before': 3690,
 'total_rows_after': 3690,
 'total_rows_removed': 0,
 'cleaning_rules_applied': 10,
 'session_result_rows_after': 40}

## Inspect cleaning impact


In [4]:
import pandas as pd

impact = pd.read_csv(outputs["paths"]["silver_cleaning_impact_summary"])
inventory = pd.read_csv(outputs["paths"]["silver_table_inventory"])
rules = pd.read_csv(outputs["paths"]["silver_cleaning_rules"])

display(inventory)
display(impact[["table_name", "rows_before", "rows_after", "rows_removed", "row_removal_pct"]])
display(rules.head(20))


,table_name,row_count,column_count,duplicate_full_rows,total_missing_cells,total_cells,missing_cell_pct
0,meetings,25,21,0,0,525,0.0000
1,sessions,123,18,0,0,2214,0.0000
2,drivers,40,16,0,1,640,0.1562
3,laps,2031,20,0,725,40620,1.7848
4,pit,62,12,0,62,744,8.3333
5,weather,303,14,0,0,4242,0.0000
6,position,927,9,0,0,8343,0.0000
7,race_control,139,15,0,534,2085,25.6115
8,session_result,40,15,0,22,600,3.6667
9,starting_grid,0,0,0,0,0,0.0000


,table_name,rows_before,rows_after,rows_removed,row_removal_pct
0,meetings,25,25,0,0.0
1,sessions,123,123,0,0.0
2,drivers,40,40,0,0.0
3,laps,2031,2031,0,0.0
4,pit,62,62,0,0.0
5,weather,303,303,0,0.0
6,position,927,927,0,0.0
7,race_control,139,139,0,0.0
8,session_result,40,40,0,0.0
9,starting_grid,0,0,0,0.0


,table_name,rule_id,rule_description,rows_before,rows_after,rows_removed,values_imputed,columns_affected,severity,rationale
0,meetings,SIL_NULL_MEETING_KEY,Drop rows with null keys: ['meeting_key'],25,25,0,0,meeting_key,high,meeting_key is required for calendar joins
1,sessions,SIL_NULL_SESSION_KEY,Drop rows with null keys: ['session_key'],123,123,0,0,session_key,high,session_key is the primary join key for all se...
2,drivers,SIL_NULL_DRIVER_KEYS,"Drop rows with null keys: ['session_key', 'dri...",40,40,0,0,"session_key,driver_number",high,session_key and driver_number required for dri...
3,laps,SIL_NULL_LAP_KEYS,"Drop rows with null keys: ['session_key', 'dri...",2031,2031,0,0,"session_key,driver_number,lap_number",high,"Lap grain requires session, driver, and lap nu..."
4,pit,SIL_NULL_PIT_KEYS,"Drop rows with null keys: ['session_key', 'dri...",62,62,0,0,"session_key,driver_number",high,Pit rows require session and driver
5,weather,SIL_NULL_WEATHER_SESSION,Drop rows with null keys: ['session_key'],303,303,0,0,session_key,high,Weather rows must link to a session
6,position,SIL_NULL_POSITION_KEYS,"Drop rows with null keys: ['session_key', 'dri...",927,927,0,0,"session_key,driver_number",high,Position updates require session and driver
7,race_control,SIL_NULL_RC_SESSION,Drop rows with null keys: ['session_key'],139,139,0,0,session_key,high,Race control messages must link to a session
8,session_result,SIL_NULL_RESULT_KEYS,"Drop rows with null keys: ['session_key', 'dri...",40,40,0,0,"session_key,driver_number",high,Final classification requires session and driver
9,starting_grid,SIL_EMPTY,No Bronze starting_grid data,0,0,0,0,NaN,low,Optional endpoint; heuristic may use sessions/...


## Missingness before vs after


In [5]:
miss_before = pd.read_csv(outputs["paths"]["silver_missingness_before"])
miss_after = pd.read_csv(outputs["paths"]["silver_missingness_after"])

display(miss_before.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))
display(miss_after.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))


,table_name,column_name,dtype,row_count,missing_count,missing_pct,non_missing_count,unique_count
79,pit,stop_duration,object,62,62,100.0000,0,0
119,race_control,qualifying_phase,object,139,139,100.0000,0,0
118,race_control,sector,float64,139,126,90.6475,13,4
113,race_control,driver_number,float64,139,101,72.6619,38,11
117,race_control,scope,object,139,84,60.4317,55,3
116,race_control,flag,object,139,84,60.4317,55,7
132,session_result,duration,float64,40,18,45.0000,22,22
63,laps,i1_speed,float64,2031,537,26.4402,1494,84
70,laps,st_speed,float64,2031,172,8.4687,1859,127
125,session_result,position,float64,40,2,5.0000,38,20


,table_name,column_name,dtype,row_count,missing_count,missing_pct,non_missing_count,unique_count
79,pit,stop_duration,object,62,62,100.0000,0,0
119,race_control,qualifying_phase,object,139,139,100.0000,0,0
118,race_control,sector,float64,139,126,90.6475,13,4
113,race_control,driver_number,float64,139,101,72.6619,38,11
117,race_control,scope,object,139,84,60.4317,55,3
116,race_control,flag,object,139,84,60.4317,55,7
132,session_result,duration,float64,40,18,45.0000,22,22
63,laps,i1_speed,float64,2031,537,26.4402,1494,84
70,laps,st_speed,float64,2031,172,8.4687,1859,127
125,session_result,position,float64,40,2,5.0000,38,20


## Duplicates & referential integrity


In [6]:
dups = pd.read_csv(outputs["paths"]["silver_duplicate_report"])
ref = pd.read_csv(outputs["paths"]["silver_referential_integrity_report"])
rejected = pd.read_csv(outputs["paths"]["silver_rejected_records_summary"])

display(dups)
display(ref)
display(rejected)


,table_name,duplicate_type,duplicate_count,duplicate_pct,stage,key_columns,duplicate_key_count,affected_rows,affected_rows_pct,status
0,meetings,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN
1,meetings,key_duplicate,NaN,NaN,before,meeting_key,0.0,0.0,0.0,checked
2,sessions,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN
3,sessions,key_duplicate,NaN,NaN,before,session_key,0.0,0.0,0.0,checked
4,drivers,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN
5,drivers,key_duplicate,NaN,NaN,before,"session_key,driver_number",0.0,0.0,0.0,checked
6,laps,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN
7,laps,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0,checked
8,pit,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN
9,pit,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0,checked


,child_table,parent_table,key_columns,child_rows,unmatched_rows,unmatched_pct,status,stage
0,drivers,sessions,session_key,2,0,0.0,checked,before
1,laps,sessions,session_key,2,0,0.0,checked,before
2,pit,sessions,session_key,2,0,0.0,checked,before
3,weather,sessions,session_key,2,0,0.0,checked,before
4,position,sessions,session_key,2,0,0.0,checked,before
5,race_control,sessions,session_key,2,0,0.0,checked,before
6,session_result,sessions,session_key,2,0,0.0,checked,before
7,starting_grid,sessions,session_key,0,0,0.0,skipped_empty_child,before
8,laps,drivers,"session_key,driver_number",40,0,0.0,checked,before
9,pit,drivers,"session_key,driver_number",38,0,0.0,checked,before


,table_name,reason,rule_id,rejected_count
